# Lab 1: Statistical Foundations — Trust and Happiness

Lab 0 was about understanding the structure of the data. Lab 1 is about measuring something inside it.
I picked trust — specifically, each country's perceived freedom from corruption — as my starting variable. My intuition was that countries where people trust their institutions would be happier. But I wanted to move from intuition to evidence.

## Loading and normalizing the dataset — unifying the yearly CSVs

Before I can analyze trust, I need to bring all five years of data together with a consistent schema, using the normalization mapping I established in Lab 0.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("..") / ".." / "data" / "raw"
OUTPUT_DIR = Path("outputs")
PLOTS_DIR = OUTPUT_DIR / "plots"
TABLES_DIR = OUTPUT_DIR / "tables"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

COLUMN_MAP = {
    "Country": "country",
    "Country or region": "country",
    "Happiness Score": "happiness_score",
    "Happiness.Score": "happiness_score",
    "Score": "happiness_score",
    "Trust (Government Corruption)": "trust",
    "Trust..Government.Corruption.": "trust",
    "Perceptions of corruption": "trust",
    "Economy (GDP per Capita)": "gdp_per_capita",
    "Economy..GDP.per.Capita.": "gdp_per_capita",
    "GDP per capita": "gdp_per_capita",
    "Health (Life Expectancy)": "life_expectancy",
    "Health..Life.Expectancy.": "life_expectancy",
    "Healthy life expectancy": "life_expectancy",
    "Freedom": "freedom",
    "Freedom to make life choices": "freedom",
    "Family": "social_support",
    "Social support": "social_support",
    "Generosity": "generosity"
}

def infer_year_from_filename(name: str) -> int | None:
    match = re.search(r"(19|20)\d{2}", name)
    return int(match.group()) if match else None

def load_all_years(data_dir: Path) -> pd.DataFrame:
    files = sorted(data_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {data_dir.resolve()}")

    frames = []
    for fp in files:
        df = pd.read_csv(fp)
        df["year"] = infer_year_from_filename(fp.name)
        df = df.rename(columns={c: COLUMN_MAP.get(c, c) for c in df.columns})

        keep = [
            "year", "country", "happiness_score", "trust",
            "gdp_per_capita", "life_expectancy", "freedom",
            "social_support", "generosity"
        ]
        existing = [c for c in keep if c in df.columns]
        df = df[existing]
        frames.append(df)

    full = pd.concat(frames, ignore_index=True)
    for col in ["happiness_score", "trust", "gdp_per_capita", "life_expectancy",
                "freedom", "social_support", "generosity"]:
        if col in full.columns:
            full[col] = pd.to_numeric(full[col], errors="coerce")
    return full

## Building a single snapshot per country — one observation each, no time-series noise

To get a clean baseline without time-series complexity, I took the latest available year per country. If a country appears in 2019, I use its 2019 row. If it only appears in 2015, I use that. This gives me one observation per country — no country biases the results by appearing multiple times.

In [2]:
df = load_all_years(DATA_DIR)
df.head()

,year,country,happiness_score,trust,gdp_per_capita,life_expectancy,freedom,social_support,generosity
0,2015,Switzerland,7.587,0.41978,1.39651,0.94143,0.66557,1.34951,0.29678
1,2015,Iceland,7.561,0.14145,1.30232,0.94784,0.62877,1.40223,0.43630
2,2015,Denmark,7.527,0.48357,1.32548,0.87464,0.64938,1.36058,0.34139
3,2015,Norway,7.522,0.36503,1.45900,0.88521,0.66973,1.33095,0.34699
4,2015,Canada,7.427,0.32957,1.32629,0.90563,0.63297,1.32261,0.45811


## Computing trust distribution and finding the extremes — who trusts their institutions most?

I needed to see what the trust scores actually look like. Computing descriptive statistics and identifying the top and bottom countries gives context to the numbers before running any regressions.

In [3]:
df_latest = (
    df.dropna(subset=["country"])
      .sort_values(["country", "year"])
      .groupby("country", as_index=False)
      .tail(1)
      .reset_index(drop=True)
)
df_latest.head()

,year,country,happiness_score,trust,gdp_per_capita,life_expectancy,freedom,social_support,generosity
0,2019,Afghanistan,3.203,0.025,0.350,0.361,0.000,0.517,0.158
1,2019,Albania,4.719,0.027,0.947,0.874,0.383,0.848,0.178
2,2019,Algeria,5.211,0.114,1.002,0.785,0.086,1.160,0.073
3,2018,Angola,3.795,0.061,0.730,0.269,0.000,1.125,0.079
4,2019,Argentina,6.086,0.050,1.092,0.881,0.471,1.432,0.066


## Step 3 - Trust Summary and Rankings
Inspect trust distribution and identify top/bottom countries.

In [4]:
trust_desc = df_latest["trust"].describe()
trust_desc

count    170.000000
mean       0.115784
std        0.099142
min        0.000000
25%        0.050500
50%        0.086500
75%        0.143750
max        0.453000
Name: trust, dtype: float64

**Observation:**
I found that the top-10 trust countries are mostly Northern European and Australasian. The bottom-10 are mostly in Africa and Eastern Europe — not a surprising result, but it set up a question I'd carry into later labs: how much of this is GDP in disguise?

## Plotting the relationship — visualizing trust against happiness

To make the relationship tangible, I generated a scatter plot of trust versus happiness and a correlation heatmap. Seeing the data visually often reveals patterns that pure numbers obscure.

**Observation:**
The Pearson correlation between trust and happiness is positive and statistically significant. The simple regression gives an R² of approximately 0.18 — meaning trust alone explains about 18% of the variance in happiness score. The regression slope means that a 0.1-point increase in the trust index corresponds to roughly +0.3 in happiness score. What surprised me is that I expected a stronger relationship. 18% explanatory power for a variable that's supposed to be a core driver of happiness felt underwhelming. That told me I'd need all six features, not just one, to build anything useful.

In [5]:
numeric_cols = [c for c in df_latest.columns if c != "country"]
corr = df_latest[numeric_cols].corr(numeric_only=True)
corr["trust"].sort_values(ascending=False)

trust              1.000000
freedom            0.424092
generosity         0.394261
happiness_score    0.340085
life_expectancy    0.225970
gdp_per_capita     0.218831
social_support     0.093273
year              -0.278818
Name: trust, dtype: float64

In [6]:
reg_df = df_latest.dropna(subset=["happiness_score", "trust"])
x = reg_df["trust"].to_numpy()
y = reg_df["happiness_score"].to_numpy()

slope, intercept = np.polyfit(x, y, 1)
y_pred = intercept + slope * x
r2 = 1 - (np.sum((y - y_pred) ** 2) / np.sum((y - y.mean()) ** 2))

slope, intercept, r2

(np.float64(3.7796794712702955),
 np.float64(4.980597541811915),
 np.float64(0.11565808499226737))

## Step 4 - Diagnostics and Plots
Visualize trust distribution, trust vs happiness, and correlations.

In [7]:
PLOTS_DIR.mkdir(exist_ok=True)

plt.figure()
sns.histplot(df_latest["trust"].dropna(), bins=20, kde=True)
plt.title("Trust Distribution (Latest Year per Country)")
plt.xlabel("trust (higher = more trust / less perceived corruption)")
plt.ylabel("count")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "lab1_plot_trust_distribution.png", dpi=300)
plt.close()

plt.figure()
sns.scatterplot(data=df_latest, x="trust", y="happiness_score")
plt.title("Trust vs Happiness (Latest Year per Country)")
plt.xlabel("trust")
plt.ylabel("happiness_score")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "lab1_plot_trust_vs_happiness.png", dpi=300)
plt.close()

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=False)
plt.title("Correlation Heatmap (Latest Year per Country)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "lab1_plot_correlation_heatmap.png", dpi=300)
plt.close()

PLOTS_DIR.resolve()

WindowsPath('C:/Users/taysir/AI-Group-DataAnalysis-2k26/project/labs/lab1/outputs/plots')

**Observation:**
Looking at the scatter plot made this visible: high-trust countries cluster at the top of the happiness scale, but there's considerable spread. Some low-trust countries are surprisingly happy; some high-trust countries underperform.

## What Comes Next

The most important finding from this lab is that trust is a real predictor of happiness, but a weak one in isolation. Lab 2 adds the statistical machinery to quantify *how confident* I should be in these findings — confidence intervals, hypothesis tests, and effect sizes.